# Construção do mapa de variáveis

Este notebook gera um arquivo JSON com o mapa completo de variáveis a partir de fontes configuráveis.

**Etapas:**
1. Carregar variáveis derivadas do arquivo de fórmulas.
2. Enriquecer com o dicionário de variáveis (tipos, valores possíveis, thresholds).
3. Validar condições contra arquivo de filtros.
4. Complementar com variáveis presentes nos quadros e ausentes nas etapas anteriores.
5. Salvar o mapa gerado (nunca sobrescreve o benchmark).
6. Comparar com o benchmark e exibir diferenças.

In [ ]:
import json

mapa_referencia = json.load(open('../data/mapa_variaveis.json', 'r'))

In [2]:
import glob as _glob
import json
from pathlib import Path
from typing import Any

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)

In [1]:
# === CONFIGURAÇÃO ===
# Qualquer variável ainda None ao final desta célula causará erro. Os valores são carregados de 
# mapa_variaveis.json

# Índices de colunas — arquivo de variáveis derivadas
COL_DERIV_CODIGO  = 1   # coluna com o código da variável
COL_DERIV_DTYPE   = 2   # coluna com o tipo de dado
COL_DERIV_NOME    = 3   # coluna com o nome/descrição
COL_DERIV_EQUACAO = 6   # coluna com a fórmula/equação

# Índices de colunas — arquivo de dicionário
COL_DIC_CODIGO = 0   # coluna com o código da variável (vazio para linhas de valor possível)
COL_DIC_DTYPE  = 4   # coluna com o tipo de dado (variável) ou código do valor (linha de valor)
COL_DIC_NOME   = 5   # coluna com o nome da variável ou descrição do valor
COL_DIC_INT    = 2   # coluna com o comprimento do inteiro
COL_DIC_FLOAT  = 3   # coluna com o comprimento do decimal
DIC_SKIP_HEAD  = 7   # número de linhas de cabeçalho a pular em cada aba

# Tokens de tipo reconhecidos na coluna COL_DIC_DTYPE para identificar linhas de variável
VALID_DTYPE_TOKENS: set[str] = {"N", "C", "D", "A"}

VARIABLE_DTYPE_MAP = {
    "N": "N",
    "numero": "Float32",
    "C": "category",
    "radiobutton": "category",
    "D": "Timestamp",
    "A": "object",
}

# Override opcional via arquivo privado não versionado
_PRIVATE = Path("../config/mapa_variaveis.private.json")
if _PRIVATE.exists():
    _cfg = json.loads(_PRIVATE.read_text(encoding="utf-8"))
    PATH_DERIVADAS         = _cfg.get("path_derivadas")
    PATH_DICIONARIOS       = _cfg.get("path_dicionarios")
    NOME_ABAS_VALIDAS      = set(_cfg.get("nomes_validos_dicionario", []))
    PATH_FILTROS           = _cfg.get("path_filtros")
    GLOB_QUADROS           = _cfg.get("glob_quadros")
    PATH_OUTPUT            = _cfg.get("path_output")
    ADD_INFO_BY_VARIABLE   = _cfg.get("add_info_by_variable", {})
    print("Configurações carregadas do arquivo privado.")
else:
    print("Arquivo de configuração privado não encontrado. Siga as instruções para criar e preencher o arquivo ../config/mapa_variaveis.private.json.")
    exit(1)

# --- Validações ---
_required = {
    "PATH_DERIVADAS":  PATH_DERIVADAS,
    "PATH_DICIONARIOS": PATH_DICIONARIOS,
    "PATH_FILTROS":    PATH_FILTROS,
    "GLOB_QUADROS":    GLOB_QUADROS,
    "PATH_OUTPUT":     PATH_OUTPUT,
}
_missing = [k for k, v in _required.items() if v is None]
assert not _missing, f"Configure as variáveis obrigatórias antes de executar: {_missing}"

_PATH_DERIVADAS  = Path(PATH_DERIVADAS)
_PATH_DICIONARIOS = [Path(p) for p in PATH_DICIONARIOS]
_PATH_FILTROS    = Path(PATH_FILTROS)
_PATH_OUTPUT     = Path(PATH_OUTPUT)

for label, p in [("Derivadas", _PATH_DERIVADAS), ("Dicionário", _PATH_DICIONARIOS), ("Filtros", _PATH_FILTROS)]:
    if isinstance(p, list):
        for sub_p in p:
            assert sub_p.exists(), f"{label} não encontrado: {sub_p}"
    else:
        assert p.exists(), f"{label} não encontrado: {p}"

_quadros_files = sorted(_glob.glob(str(GLOB_QUADROS)))
assert _quadros_files, f"Nenhum arquivo de quadros encontrado com o padrão: {GLOB_QUADROS}"

# Validação dos paths
print(f"Derivadas     : {_PATH_DERIVADAS.name}")
print(f"Dicionário    : {', '.join([p.name for p in _PATH_DICIONARIOS])}")
print(f"Filtros       : {_PATH_FILTROS.name}")
print(f"Quadros       : {len(_quadros_files)} arquivo(s)")
print(f"Saída         : {_PATH_OUTPUT}")

NameError: name 'Path' is not defined

## Funções Auxiliares

In [4]:

def load_additional_info(path: str | Path) -> dict[str, dict]:
    """Loads a product info JSON and indexes entries by product code."""    
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    response = {}
    for p in data.get("infoProdutos", []) + data.get("Produtos", []):
        codigo = p.get("codigoProduto") or p.get("CodigoProduto")
        if codigo is not None:
            #itera sobre todas as entradas chaves de p deixando a primeira letra minuscula
            p = {k[0].lower() + k[1:] if k else k: v for k, v in p.items()}
            response[str(codigo)] = p

    return response

def parse_derivadas(
    path: Path,
    col_codigo: int,
    col_dtype: int,
    col_nome: int,
    col_equacao: int,
) -> tuple[dict[str, dict], list[str]]:
    """Reads the derived-variables file.
    Returns (variable_map, list_of_duplicate_codes)."""
    df = pd.read_excel(path, header=None)
    variable_map: dict[str, dict] = {}
    duplicates: list[str] = []

    for _, row in df.iterrows():
        code_raw = row[col_codigo]
        if pd.isna(code_raw) or not str(code_raw).strip():
            continue
        code = str(code_raw).strip()
        if code in variable_map:
            duplicates.append(code)
            continue
        equacao = row[col_equacao]
        variable_map[code] = {
            "name":           row[col_nome],
            "dtype":          VARIABLE_DTYPE_MAP.get(row[col_dtype], ""),
            "type":           "condition" if isinstance(equacao, str) and equacao.strip() else "value",
            "condition":      equacao if isinstance(equacao, str) else None,
            "possible_values": [],
        }

    return variable_map, duplicates


def parse_dicionario_sheets(
    path: Path,
    col_codigo: int,
    col_dtype: int,
    col_nome: int,
    col_int_len: int,
    col_float_len: int,
    skip_head: int,
    valid_dtypes: set[str],
    add_info_by_variable: dict[str, str],
) -> dict[str, dict]:
    """Reads all sheets of the dictionary XLS.
    Uses each sheet name as the reference label for its variables.
    Returns a dict of variable entries keyed by variable code."""
    xl = pd.ExcelFile(path)
    entries: dict[str, dict] = {}
    _cache: dict[str, dict] = {}  # path → loaded additional_info

    for sheet in xl.sheet_names:
        if sheet not in NOME_ABAS_VALIDAS:
            print(f"Aba '{sheet}' ignorada por não estar na lista de nomes válidos.")
            continue
        
        df = pd.read_excel(xl, header=None, sheet_name=sheet, skiprows=skip_head)
        reference = sheet
        last_var: str | None = None
        current_add_info: dict = {}

        for _, row in df.iterrows():
            code_raw = row[col_codigo]
            is_empty = pd.isna(code_raw) or (
                isinstance(code_raw, str) and 
                not code_raw.strip()
            )

            if not is_empty:
                # Identify valid variable rows by the dtype token
                dtype_raw = row[col_dtype]
                dtype_token = str(dtype_raw).strip().upper() if not pd.isna(dtype_raw) else ""
                if dtype_token not in valid_dtypes:
                    continue  # metadata / header row — skip

                code = str(code_raw).strip().replace("X_", "")
                last_var = code

                info_path = add_info_by_variable.get(code)
                if info_path:
                    if info_path not in _cache:
                        _cache[info_path] = load_additional_info(info_path)
                    if "EFET_SILVI" in info_path:
                        print(f"Informações adicionais para {code} carregadas de {info_path}:")
                        print(_cache[info_path])
                    
                    current_add_info = _cache[info_path]
                else:
                    current_add_info = {}

                parsed_dtype_token = VARIABLE_DTYPE_MAP.get(dtype_token, "")
                if parsed_dtype_token == "N":
                    int_len = row[col_int_len]
                    float_len = row[col_float_len]
                    if not pd.isna(int_len) and not str(int_len).strip() == "":
                        if int(int_len) > 9:
                            parsed_dtype_token = "Int64"
                        elif int(int_len) > 0:
                            parsed_dtype_token = "Int32"
                        
                        # if int(int_len) >= 19:
                        #     print(f"Variável {code} tem comprimento inteiro declarado como {int_len}, o que pode exceder a capacidade de Int64. Verifique se isso é correto.")
                    
                    if not pd.isna(float_len) and not str(float_len).strip() == "":
                        if int(float_len) + int(int_len) > 6:
                            parsed_dtype_token = "Float64"
                        else:
                            parsed_dtype_token = "Float32"
                        
                        # if int(float_len) + int(int_len) >= 16:
                        #     print(f"Variável {code} pode exceder a capacidade de Float64. Verifique se isso é correto.")

                entries[code] = {
                    "name":            row[col_nome],
                    "dtype":           parsed_dtype_token,
                    "type":            "value",
                    "possible_values": [],
                    "condition":       None,
                    "reference":       reference,
                }

            else:

                if last_var is None:
                    continue
                val_code_raw = row[col_dtype]
                if pd.isna(val_code_raw):
                    val_code_raw = "Vazio"
                    # continue
                
                val_desc_raw = row[col_nome]
                cod_str = str(val_code_raw).strip()
                desc_str = str(val_desc_raw) if not pd.isna(val_desc_raw) else ""

                thresholds: dict = {}
                prod = current_add_info.get(cod_str, {})
                # if info_path and "EFET_SILVI" in info_path:
                #     print(f"Processando variável {last_var} - valor possível '{cod_str}':")
                #     print(f"  Descrição: '{desc_str}'")
                #     print(prod)
                #     print(current_add_info)
                if prod:
                    thresholds = {
                        "min":        prod.get("limiteInferior", ""),
                        "max":        prod.get("limiteSuperior", ""),
                        "minYield":   prod.get("rendimentoMinimo", ""),
                        "maxYield":   prod.get("rendimentoMaximo", ""),
                        "minDensity": prod.get("densidadeMinima", ""),
                        "maxDensity": prod.get("densidadeMaxima", ""),
                        "unit":       prod.get("unidadeDeMedida", ""),
                    }

                if last_var in entries:
                    entries[last_var]["possible_values"].append({
                        "cod":         cod_str,
                        "description": desc_str,
                        "thresholds":  thresholds,
                    })

    return entries


def validate_filtros(variable_map: dict, filtros_path: Path) -> dict[str, Any]:
    """Checks which variables referenced in the filtros file are present in variable_map."""
    with open(filtros_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    filtros = data.get("Filtros", [])
    not_found = sorted({e["CodigoVariavel"] for e in filtros if e.get("CodigoVariavel") not in variable_map})
    return {
        "total_filtros": len(filtros),
        "variables_not_found": not_found,
        "not_found_count": len(not_found),
    }


def complement_from_quadros(
    variable_map: dict,
    quadros_files: list[str],
) -> list[str]:
    """Adds variables from quadro JSON files that are not yet in variable_map.
    Returns list of added codes."""
    added: list[str] = []

    for file_path in quadros_files:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        for code in data.get("VariaveisDerivadas", []):
            code = code.replace("X_", "")
            if code and code not in variable_map:
                variable_map[code] = {
                    "name": "Sem titulo", "dtype": "", "type": "value",
                    "possible_values": [], "condition": None,
                }

                added.append(code)

        for section_key in ("Quesitos", "QuesitoProdutos", "QuesitoProdutosAgrupados", "ListaProdutos"):
            for quesito in data.get(section_key, []):
                for pergunta in quesito.get("Perguntas", []):
                    code = pergunta.get("CodigoVariavel", "").strip().replace("X_", "")
                    if not code:
                        continue

                    if code in variable_map:
                        if variable_map[code].get("name") in (None, "", "Sem titulo"):
                            variable_map[code]["name"] = pergunta.get("Titulo", "Sem titulo")
                        if variable_map[code].get("condition") in (None, ""):
                            variable_map[code]["condition"] = pergunta.get("CondicaoDeExibicao", None)
                        continue

                    variable_map[code] = {
                        "name":            pergunta.get("Titulo", "Sem titulo"),
                        "dtype":           VARIABLE_DTYPE_MAP.get(pergunta.get("Tipo", ""), ""),
                        "type":            pergunta.get("TipoVariavel", "value"),
                        "possible_values": [],
                        "condition":       pergunta.get("CondicaoDeExibicao", None),
                        "reference":       data.get("Grupo", "").replace("T_CRIT_", "").replace("_", " "),
                    }

                    added.append(code)

    return added


def diff_maps(new_map: dict, benchmark: dict) -> dict[str, Any]:
    """Compares new_map against benchmark. Returns a dict with change categories."""
    new_keys  = set(new_map.keys())
    bench_keys = set(benchmark.keys())

    dtype_changed: list[dict] = []
    condition_changed: list[dict] = []
    pv_count_changed: list[dict] = []
    pv_thresholds_changed: list[tuple] = []

    for code in new_keys & bench_keys:
        ne, be = new_map[code], benchmark[code]
        # Combinações de condições validas:
        # - "C" vs "category"
        # - "A" vs "object"
        # - "Int32"/"Int64"/"Float32"/"Float64" vs "N" (não é possível inferir o tipo numérico exato apenas pela declaração "N", então consideramos todas as combinações como válidas)
        # - Se o benchmark for Nan ou vazio, qualquer tipo é aceito (isso é tratado na comparação direta, pois str(NaN) e str(None) resultam em "nan" e "None", respectivamente)
        is_valid_dtype_combo = (
            (str(ne.get("dtype", "")) == "C" and str(be.get("dtype", "")) == "category") or
            (str(ne.get("dtype", "")) == "category" and str(be.get("dtype", "")) == "C") or
            (str(ne.get("dtype", "")) == "A" and str(be.get("dtype", "")) == "object") or
            (str(ne.get("dtype", "")) == "object" and str(be.get("dtype", "")) == "A") or
            (str(ne.get("dtype", "")) in {"Int32", "Int64", "Float32", "Float64"} and str(be.get("dtype", "")) == "N") or
            (pd.isna(be.get("dtype")) or str(be.get("dtype", "")).strip() == "")
        )
        if str(ne.get("dtype", "")) != str(be.get("dtype", "")) and not is_valid_dtype_combo:
            dtype_changed.append({"code": code, "new": ne.get("dtype"), "benchmark": be.get("dtype")})
        
        # Comparação de condições, tratando "None" e "nan" como equivalentes a vazio
        ne_condition = str(ne.get("condition", ""))
        be_condition = str(be.get("condition", ""))
        ne_condition = ne_condition if ne_condition != "None" and ne_condition != "nan" else ""
        be_condition = be_condition if be_condition != "None" and be_condition != "nan" else ""
        if ne_condition != be_condition:
            condition_changed.append({
                "code": code,
                "new": ne_condition[:100],
                "benchmark": be_condition[:100],
            })
        
        # Compara contagem de possíveis valores, sem entrar no detalhe de quais são
        n_pv = len(ne.get("possible_values", []))
        b_pv = len(be.get("possible_values", []))
        if n_pv != b_pv and b_pv > 0:  # mudanças de contagem são relevantes apenas se o benchmark tinha PVs
            pv_count_changed.append({"code": code, "new_count": n_pv, "benchmark_count": b_pv})
        
        # Comparando diretamente os valores possiveis para avaliar os thresholds
        # if code == "V39010100":
        #     print([pv for pv in ne.get("possible_values", [])])
        #     print({(pv.get("cod"), tuple(sorted(pv.get("thresholds", {}).items()))) for pv in ne.get("possible_values", [])})
        #     print({(pv.get("cod"), tuple(sorted(pv.get("thresholds", {}).items()))) for pv in be.get("possible_values", [])})
        ne_pvs = {(pv.get("cod"), tuple(sorted(pv.get("thresholds", {}).items()))) for pv in ne.get("possible_values", [])}
        be_pvs = {(pv.get("cod"), tuple(sorted(pv.get("thresholds", {}).items()))) for pv in be.get("possible_values", [])}
        
        if ne_pvs != be_pvs and len(be_pvs) > 0:  # mudanças de thresholds são relevantes apenas se o benchmark tinha PVs
            pv_thresholds_changed.append({
                "code": code,
                "new_pvs": sorted(ne_pvs),
                "benchmark_pvs": sorted(be_pvs),
            })

    return {
        "added":             sorted(new_keys - bench_keys),
        "removed":           sorted(bench_keys - new_keys),
        "dtype_changed":     dtype_changed,
        "condition_changed": condition_changed,
        "pv_count_changed":  pv_count_changed,
        "pv_thresholds_changed": pv_thresholds_changed,
    }

## Fluxo de Criação do Mapa de Variáveis

### Fase 1 - Variáveis derivadas

In [5]:
variable_map, deriv_duplicates = parse_derivadas(
    _PATH_DERIVADAS,
    col_codigo=COL_DERIV_CODIGO,
    col_dtype=COL_DERIV_DTYPE,
    col_nome=COL_DERIV_NOME,
    col_equacao=COL_DERIV_EQUACAO,
)

print(f"Variáveis carregadas da fase 1 : {len(variable_map)}")
if deriv_duplicates:
    print(f"Duplicatas ignoradas       : {len(deriv_duplicates)}")

print(variable_map.get("V40020600"))

Variáveis carregadas da fase 1 : 354
None


### Fase 2 - Dicionários de Variáveis

In [6]:
def merge_possible_values(possible_values1: list[dict], possible_values2: list[dict]) -> list[dict]:
    """Combina duas listas de possíveis valores, evitando duplicatas com base no código do valor."""
    combined = {pv["cod"]: pv for pv in possible_values1}
    for pv in possible_values2:
        if pv["cod"] not in combined:
            combined[pv["cod"]] = pv
    return list(combined.values())

# Processando os dicionários: cada entrada de variável é enriquecida com dtype, possible_values e reference do dicionário;
dict_entries = []
for path in _PATH_DICIONARIOS:
    dict_entries.append(parse_dicionario_sheets(
        path,
        col_codigo=COL_DIC_CODIGO,
        col_dtype=COL_DIC_DTYPE,
        col_nome=COL_DIC_NOME,
        col_int_len=COL_DIC_INT,
        col_float_len=COL_DIC_FLOAT,
        skip_head=DIC_SKIP_HEAD,
        valid_dtypes=VALID_DTYPE_TOKENS,
        add_info_by_variable=ADD_INFO_BY_VARIABLE,
    ))

# Mescla: variáveis da fase 1 recebem possible_values e reference do dicionário;
# variáveis novas (não derivadas) são adicionadas diretamente.
only_in_dict = 0
enriched = 0
for entry in dict_entries:
    for code, entry in entry.items():
        if code == "V02210000":
            print("Debug: processando código V02210000")
            print("Entrada do dicionário:", entry)
        
        if code in variable_map:
            variable_map[code]["dtype"]           = entry["dtype"]
            variable_map[code]["possible_values"] = merge_possible_values(variable_map[code]["possible_values"],
                                                                          entry["possible_values"])
            variable_map[code]["reference"]       = entry["reference"]
            enriched += 1
        else:
            variable_map[code] = entry
            only_in_dict += 1

print(f"Entradas do dicionário         : {len(dict_entries)}")
print(f"  → enriqueceram fase 1        : {enriched}")
print(f"  → adicionadas como novas     : {only_in_dict}")
print(f"Total após fase 2              : {len(variable_map)}")

print(variable_map.get("V40020600"))

Informações adicionais para V39010100 carregadas de /home/carlos_rocha/Documentos/IBGE/ibge-censo-agro-research/data/dados_censo_2017/Metadata/informacoes_adicionais/QD39_EFET_SILVICU.json:
{'501': {'codigoProduto': 501, 'nomeProduto': 'AC�CIA NEGRA - P�S', 'limiteInferior': '', 'limiteSuperior': '', 'rendimentoMinimo': '', 'rendimentoMaximo': '', 'densidadeMinima': 50, 'densidadeMaxima': 10000, 'unidadeDeMedida': 'P�S'}, '502': {'codigoProduto': 502, 'nomeProduto': 'ALGAROBEIRA - P�S', 'limiteInferior': '', 'limiteSuperior': '', 'rendimentoMinimo': '', 'rendimentoMaximo': '', 'densidadeMinima': 20, 'densidadeMaxima': 6000, 'unidadeDeMedida': 'P�S'}, '503': {'codigoProduto': 503, 'nomeProduto': 'BAMBU (TAQUARA) - P�S', 'limiteInferior': '', 'limiteSuperior': '', 'rendimentoMinimo': '', 'rendimentoMaximo': '', 'densidadeMinima': 10, 'densidadeMaxima': 2000, 'unidadeDeMedida': 'P�S'}, '504': {'codigoProduto': 504, 'nomeProduto': 'BRACATINGA - P�S', 'limiteInferior': '', 'limiteSuperior':

In [7]:
filtros_result = validate_filtros(variable_map, _PATH_FILTROS)

print(f"Total de entradas nos filtros  : {filtros_result['total_filtros']}")
print(f"Variáveis não encontradas      : {filtros_result['not_found_count']}")
if filtros_result["variables_not_found"]:
    print("Códigos ausentes no mapa:")
    display(pd.Series(filtros_result["variables_not_found"], name="codigo").to_frame())

Total de entradas nos filtros  : 30
Variáveis não encontradas      : 0


In [8]:
added_from_quadros = complement_from_quadros(variable_map, _quadros_files)

print(f"Total após fase 4              : {len(variable_map)}")
print(f"Variáveis adicionadas          : {len(added_from_quadros)}")

Total após fase 4              : 980
Variáveis adicionadas          : 13


In [9]:
_PATH_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
_PATH_OUTPUT.write_text(
    json.dumps(variable_map, indent=4, ensure_ascii=False),
    encoding="utf-8",
)

print(f"Mapa salvo em: {_PATH_OUTPUT.resolve()}")
print(f"Total de variáveis : {len(variable_map)}")

# Contagens por referência e por tipo
refs = {}
types = {}
for entry in variable_map.values():
    ref = entry.get("reference", "(sem referência)")
    typ = entry.get("type", "(sem tipo)")
    refs[ref]   = refs.get(ref, 0) + 1
    types[typ]  = types.get(typ, 0) + 1

print("\nPor referência:")
display(pd.DataFrame.from_dict(refs, orient="index", columns=["count"]).sort_values("count", ascending=False))
print("\nPor tipo:")
display(pd.DataFrame.from_dict(types, orient="index", columns=["count"]))

Mapa salvo em: /home/carlos_rocha/Documentos/IBGE/ibge-censo-agro-research/data/mapa_variaveis_new.json
Total de variáveis : 980

Por referência:


,count
ESTABELEC AGRO,598
PECUARIA,251
LAVOURA PERMANENTE,31
LAVOURA TEMPORARIA,19
AGROINDUSTRIA,16
(sem referência),14
EXTRACAO VEGETAL,14
HORTICULTURA,12
PRODUTOS DA SILVICULTURA,11
EFETIVOS DA SILVICULTURA,9



Por tipo:


,count
condition,330
value,650


In [10]:
diff = diff_maps(variable_map, mapa_referencia)

# print(f"Benchmark         : {_PATH_BENCHMARK.name}  ({len(benchmark_map)} variáveis)")
print(f"Novo mapa         : {_PATH_OUTPUT.name}  ({len(variable_map)} variáveis)")
print(f"Adicionadas       : {len(diff['added'])}")
print(f"Removidas         : {len(diff['removed'])}")
print(f"dtype alterado    : {len(diff['dtype_changed'])}")
print(f"condition alterada: {len(diff['condition_changed'])}")
print(f"qtd. valores diff : {len(diff['pv_count_changed'])}")
print(f"thresholds diff   : {len(diff['pv_thresholds_changed'])}")

if diff["dtype_changed"]:
    print("\nAlterações de dtype:")
    display(pd.DataFrame(diff["dtype_changed"]))

if diff["condition_changed"]:
    print("\nAlterações de condition:")
    display(pd.DataFrame(diff["condition_changed"]))

if diff["pv_count_changed"]:
    print("\nAlterações na quantidade de valores possíveis:")
    display(pd.DataFrame(diff["pv_count_changed"]))

if diff["pv_thresholds_changed"]:
    print("\nAlterações nos thresholds dos valores possíveis:")
    display(pd.DataFrame(diff["pv_thresholds_changed"]))

Novo mapa         : mapa_variaveis_new.json  (980 variáveis)
Adicionadas       : 1
Removidas         : 1
dtype alterado    : 24
condition alterada: 109
qtd. valores diff : 1
thresholds diff   : 5

Alterações de dtype:


,code,new,benchmark
0,VX22020101,,campo_informativo_diferenca
1,VX03100000,,campo_informativo_diferenca
2,V34020400,category,C
3,V40020501,Float64,Float32
4,VW34020501,Float64,Float32
5,V34020800,Float64,Float32
6,VW14060000,Float64,Float32
7,VX19020101,,campo_informativo_diferenca
8,V34020700,category,C
9,V37020501,Float64,Float32



Alterações de condition:


,code,new,benchmark
0,V07020102,V07020101 > 0,
1,V04020000,V04020001 == 2,
2,V35021000,X_V35020602 == 2,
3,V12010103,V12010000 == 2,
4,V45010800,V05180100 == 2,
5,V02120100,V02120000 == 2,
6,V05010101,V05010100 == 2,
7,V05020500,V05020100 == 2,
8,V12030100,V12010000 == 2 && V12010101 == 2 && V12010104 != 2,
9,V05340109,V05340003 == 2,



Alterações na quantidade de valores possíveis:


,code,new_count,benchmark_count
0,VW04280000,12,11



Alterações nos thresholds dos valores possíveis:


,code,new_pvs,benchmark_pvs
0,V01170501,"[(, ()), (1, ()), (2, ()), (3, ()), (4, ()), (5, ()), (6, ())]","[( , ()), (1, ()), (2, ()), (3, ()), (4, ()), (5, ()), (6, ())]"
1,VW04280000,"[(01, ()), (02, ()), (03, ()), (04, ()), (05, ()), (06, ()), (07, ()), (08, ...","[(01, ()), (02, ()), (03, ()), (04, ()), (05, ()), (06, ()), (07, ()), (08, ..."
2,V02230000,"[(1, ()), (2, ()), (Vazio, ())]","[(1, ()), (2 , ()), (Vazio, ())]"
3,V02230001,"[(1, ()), (2, ()), (Vazio, ())]","[(1, ()), (2 , ()), (Vazio, ())]"
4,V39010100,"[(501, (('max', ''), ('maxDensity', 10000), ('maxYield', ''), ('min', ''), (...","[(501, (('max', ''), ('maxDensity', 10000), ('maxYield', ''), ('min', ''), (..."
